# Exercise 04 : A Multi-Agent Visit Planner

- A fixed workflow runs two specialists in parallel ; `Send` gives each one its own task
- Each specialist is its own agent, with its own tool and its own message list
- A checker written in code checks specified properties without a model call
- A second design has to earn its calls and tokens against one agent

You will plan one visit to site S04. Devices come from the database. A parts
agent and a manuals agent run in parallel. A checker reads the plan, then one
agent does the same task. The Aurelia records are made up.

Worked examples to revisit : module 05, demo03b · router-parallel, demo06b · workflows-parallel,
demo07 · maker-checker, demo09 · what-each-pattern-costs.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## Step 1 : Tools

`run_sql` and `search_manuals` read the Aurelia database and manuals.

In [2]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)


counter = Counter()
metered = make_llm(callbacks=[counter], max_tokens=600, max_retries=0, timeout=120)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = {"pattern": label, "calls": counter.calls, "input_tokens": counter.input,
               "output_tokens": counter.output, "seconds": round(time.perf_counter() - start, 1)}
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: sites(site_id, name, city, contract),
    devices(serial, model, site_id, installed, firmware),
    parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, serial, opened, status, error_code, summary, engineer_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(15))


@tool
def search_manuals(query: str) -> str:
    """Find Aurelia manuals and bulletins. Returns the opening of the two best matches."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:2]
    return "\n\n".join(f"[{name}]\n{PAGES[name][:800]}" for name in ranked)

In [3]:
listed = run_sql.invoke({"query": "SELECT serial FROM devices WHERE site_id = 'S04'"})
blocked = run_sql.invoke({"query": "DELETE FROM devices"})
print("devices :", listed)
print("blocked :", blocked)
ok = "MR150-2024-0098" in listed and "UX20-2025-2311" in listed and blocked.startswith("Only SELECT")
print("check :", "pass" if ok else "fail")

devices : [["MR150-2024-0098"], ["UX20-2025-2311"]]
blocked : Only SELECT statements are allowed.
check : pass


## Step 2 : Checker

Return a list of problems, empty when the plan passes. Every device at the
site appears by serial number. Every part number such as `MR-CH-02` exists
in `parts`. A part with stock 0 has "out of stock" on the same line.

In [4]:
import re


PART = re.compile(r"\b[A-Z]{2}-[A-Z]{2}-\d{2}\b")


def check_plan(plan: str, site_id: str) -> list[str]:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        serials = [s for (s,) in db.execute("SELECT serial FROM devices WHERE site_id = ?", (site_id,))]
        stock = dict(db.execute("SELECT part_no, stock FROM parts"))
    problems = [f"device {s} is missing" for s in serials if s not in plan]
    for line in plan.splitlines():
        for part in PART.findall(line):
            if part not in stock:
                problems.append(f"part {part} does not exist")
            elif stock[part] == 0 and "out of stock" not in line.lower():
                problems.append(f"part {part} has no stock and the plan does not say so")
    return sorted(set(problems))

In [5]:
GOOD = """Visit plan S04
- MR150-2024-0098 : WO-9003 needs cold head MR-CH-02, out of stock at Venlo
- UX20-2025-2311 : probe cleaning check, bring fan filter UX-FF-03"""
BAD = """Visit plan S04
- MR150-2024-0098 : bring cold head MR-CH-02 and filter MR-XX-99"""
print("good :", check_plan(GOOD, "S04"))
print("bad  :", check_plan(BAD, "S04"))
assert check_plan(GOOD, "S04") == [], "the good plan should pass"
assert len(check_plan(BAD, "S04")) == 3, "the bad plan has three problems"
print("check : pass")

good : []
bad  : ['device UX20-2025-2311 is missing', 'part MR-CH-02 has no stock and the plan does not say so', 'part MR-XX-99 does not exist']
check : pass


## Step 3 : Parallel specialists

Complete the manuals question : ask which maintenance intervals and service
bulletins apply to the devices in `state["devices"]`. Then complete the two
`Send`s, the join into `assemble`, and `after_check`. Each specialist is an
agent built from a model node and a tool node. The parts agent receives its
query in `parts_question`. This is the Parallel Agents workflow of module 05 :
no classifier chooses the specialists, because every visit needs both.

In [6]:
class Visit(TypedDict):
    site_id: str
    devices: str
    sections: Annotated[list[str], operator.add]
    plan: str
    problems: list[str]
    rounds: int


class Task(TypedDict):
    question: str


def agent_by_hand(model, tools, prompt):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(MessagesState)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    for message in reversed(result["messages"]):
        if isinstance(message, AIMessage) and message.content and not message.tool_calls:
            return str(message.content)
    return str(result["messages"][-1].content)


def specialist(name, worker):
    def node(state: Task) -> dict:
        try:
            answer = last_text(worker.invoke(ask(state["question"]), {"recursion_limit": 10}))
        except GraphRecursionError:
            answer = "(no answer : the specialist hit its step limit)"
        print(name, ":", answer[:300])
        return {"sections": [name + " :\n" + answer]}
    return node


PLAN_RULES = ("One bullet per device, starting with its serial number. Bring only the part that the open "
              "work order's error and summary point to. Write 'out of stock' next to any part with stock 0.")


def devices(state: Visit) -> dict:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute(
            "SELECT serial, model, firmware FROM devices WHERE site_id = ?", (state["site_id"],)).fetchall()
    return {"devices": "\n".join(f"{serial} {model} firmware {firmware}" for serial, model, firmware in rows)}


def manuals_question(state: Visit) -> str:
    return ("For each device model, search its maintenance page and service bulletins. Which maintenance intervals and service bulletins apply to these devices?\n"
            + state["devices"])


def parts_question(state: Visit) -> str:
    return (
        "List the open work orders at the site and the spare parts for each device's model. "
        "A status other than closed still counts. Run this SELECT once and report every row:\n"
        "SELECT w.wo_id, w.serial, w.status, w.error_code, w.summary, p.part_no, p.name, p.stock, p.depot "
        "FROM work_orders w JOIN devices d ON d.serial = w.serial "
        "JOIN parts p ON p.model = d.model "
        f"WHERE d.site_id = '{state['site_id']}' AND w.status != 'closed'")


def dispatch(state: Visit):
    return [
        Send("parts", {"question": parts_question(state)}),
        Send("manuals", {"question": manuals_question(state)}),
    ]


def assemble(state: Visit) -> dict:
    feedback = ""
    if state["problems"]:
        feedback = "\nFix these problems found by the checker :\n" + "\n".join(state["problems"])
    plan = metered.invoke(
        f"Write a visit plan for site {state['site_id']}. {PLAN_RULES}\n\nDevices :\n{state['devices']}\n\n"
        + "\n\n".join(state["sections"]) + feedback).content
    return {"plan": plan, "rounds": state["rounds"] + 1}


def check(state: Visit) -> dict:
    found = check_plan(state["plan"], state["site_id"])
    print(f"round {state['rounds']} :", found or "passed")
    return {"problems": found}


def after_check(state: Visit):
    return "assemble" if state["problems"] and state["rounds"] < 2 else END


parts_worker = agent_by_hand(metered, [run_sql], "Call run_sql once. Then answer from those rows and stop.")
manuals_worker = agent_by_hand(metered, [search_manuals],
                               "For each device model, call search_manuals twice : the model with 'maintenance', "
                               "then the model with 'service bulletin'. "
                               "Then answer, cite the file names, and stop.")

b = StateGraph(Visit)
b.add_node("devices", devices)
b.add_node("parts", specialist("parts", parts_worker), input_schema=Task)
b.add_node("manuals", specialist("manuals", manuals_worker), input_schema=Task)
b.add_node("assemble", assemble)
b.add_node("check", check)
b.add_edge(START, "devices")
b.add_conditional_edges("devices", dispatch, ["parts", "manuals"])
b.add_edge(["parts", "manuals"], "assemble")
b.add_edge("assemble", "check")
b.add_conditional_edges("check", after_check, ["assemble", END])
planner = b.compile()
print("planner built")

planner built


In [7]:
sample = {"site_id": "S04", "devices": "MR150-2024-0098 MR-150 firmware 12.0",
          "sections": [], "plan": "", "problems": ["device missing"], "rounds": 1}
sent = {item.node: item.arg["question"] for item in dispatch(sample)}
manual_q = sent.get("manuals", "").lower()
routes_ok = (set(sent) == {"parts", "manuals"} and "MR150-2024-0098" in sent.get("manuals", "")
             and "interval" in manual_q and "bulletin" in manual_q)
again_ok = after_check(sample) == "assemble"
stop_ok = after_check({**sample, "problems": []}) == END and after_check({**sample, "rounds": 2}) == END
print("nodes :", sorted(sent))
print("check :", "pass" if routes_ok and again_ok and stop_ok else "fail")

nodes : ['manuals', 'parts']
check : pass


## Step 4 : Cost

Plan S04, then give one agent both tools and the same task.

In [8]:
def run_planner(site):
    return planner.invoke(
        {"site_id": site, "devices": "", "sections": [], "plan": "", "problems": [], "rounds": 0},
        {"recursion_limit": 12})


planner_result = measured("planner", lambda: run_planner("S04"))
print(planner_result.get("plan", "")[:800])
print("planner problems :", planner_result.get("problems"), "| rounds :", planner_result.get("rounds"))
print("sections :", [section.split(" :")[0] for section in planner_result.get("sections", [])])

single = create_agent(
    metered,
    tools=[run_sql, search_manuals],
    system_prompt="Write a visit plan for site S04. " + PLAN_RULES
    + " work_orders has no site_id; join it to devices on serial. A status other than closed still counts. "
    + "Use run_sql and search_manuals, then stop.")
single_result = measured(
    "one agent",
    lambda: single.invoke({"messages": [HumanMessage("Plan the visit to site S04.")]}, {"recursion_limit": 8}))
answer = last_text(single_result)
print(answer[:800])
print("one agent problems :", check_plan(answer, "S04"))

manuals : - **MR-150** (MR150-2024-0098, firmware 12.0): Replace the helium compressor filter every **10,000 operating hours**; inspect the cold head **yearly**; check the RF coil **every 6 months**. **SB-MR-2025-07** changes the filter interval to **8,000 hours** only if the equipment room is above 25°C for 


parts : | Work order | Serial | Status | Error code | Summary | Part no. | Part | Stock | Depot |
|---|---|---|---|---|---|---|---:|---|
| WO-9003 | MR150-2024-0098 | waiting for part | E-MR-202 | Cold head temperature high | MR-CH-02 | cold head | 0 | Venlo |
| WO-9003 | MR150-2024-0098 | waiting for part 


round 1 : passed
metrics : {"pattern": "planner", "calls": 5, "input_tokens": 2717, "output_tokens": 645, "seconds": 14.8}
- **MR150-2024-0098** — Bring **MR-CH-02 cold head** (out of stock). The open work order reports high cold-head temperature.
- **UX20-2025-2311** — No open work order or part is listed; bring no part.
planner problems : [] | rounds : 1
sections : ['parts', 'manuals']


metrics : {"pattern": "one agent", "calls": 4, "input_tokens": 4374, "output_tokens": 399, "seconds": 9.9}
- **MR150-2024-0098** — Open work order WO-9003 (waiting for part), error E-MR-202: cold head temperature high. Bring **MR-CH-02 cold head** — **out of stock**. Manual calls for an MR specialist and a planned ramp-down.
- **UX20-2025-2311** — No open work order; bring no part.
one agent problems : []


In [9]:
ok = bool(planner_result.get("plan")) and len(metrics) >= 2 and all(row["calls"] >= 1 for row in metrics[-2:])
print("check :", "pass" if ok else "fail")
for row in metrics[-2:]:
    print(row["pattern"], row["calls"], "calls", row["input_tokens"] + row["output_tokens"], "tokens")

check : pass
planner 5 calls 3362 tokens
one agent 4 calls 4773 tokens


## If time permits

- Plan site S01 with `run_planner` and check that the part matches the CT-400's work order
- Remove "Write 'out of stock' ..." from `PLAN_RULES`, run the planner again and read round 2
- Add a fourth check : every CT-400 on firmware below 5.2 gets the 12-month fan interval
- Replace the parallel dispatch with a handoff. The transfer tool refuses a transfer that
  shares its turn with other tool calls and forwards the full worker history, as in
  demo04b · handoffs-fixed